## **년도별 아파트 거래내역 CSV 병합**

#### **0. 라이브러리 임포트**

In [1]:
import pandas as pd
import os

print("라이브러리 임포트 완료")

라이브러리 임포트 완료


#### **1. 파일 경로 설정 및 CSV 목록 수집**

In [2]:
# 1. 검단_불로동 아파트 실거래가 CSV 파일 경로 지정
folder_path = r"C:\3_1_DataMining\팀 프로젝트\검단_불로동"

# 2. 폴더 내 CSV 파일 목록 수집 (결과 파일 제외)
all_files = []
for f in os.listdir(folder_path):
    ext = os.path.splitext(f)[-1].lower()
    if ext == '.csv' and "geomdan_bullo_merged" not in f:
        full_path = os.path.join(folder_path, f)
        all_files.append(full_path)

print(f"수집된 CSV 파일 수: {len(all_files)}개")
for f in all_files:
    print(f"  - {os.path.basename(f)}")

수집된 CSV 파일 수: 21개
  - 아파트(매매)_실거래가_20260531230344.csv
  - 아파트(매매)_실거래가_20260531230404.csv
  - 아파트(매매)_실거래가_20260531230424.csv
  - 아파트(매매)_실거래가_20260531230450.csv
  - 아파트(매매)_실거래가_20260531230511.csv
  - 아파트(매매)_실거래가_20260531230525.csv
  - 아파트(매매)_실거래가_20260531230544.csv
  - 아파트(매매)_실거래가_20260531230556.csv
  - 아파트(매매)_실거래가_20260531230608.csv
  - 아파트(매매)_실거래가_20260531230624.csv
  - 아파트(매매)_실거래가_20260531230637.csv
  - 아파트(매매)_실거래가_20260531230651.csv
  - 아파트(매매)_실거래가_20260531230703.csv
  - 아파트(매매)_실거래가_20260531230716.csv
  - 아파트(매매)_실거래가_20260531230730.csv
  - 아파트(매매)_실거래가_20260531230747.csv
  - 아파트(매매)_실거래가_20260531230806.csv
  - 아파트(매매)_실거래가_20260531230819.csv
  - 아파트(매매)_실거래가_20260531230830.csv
  - 아파트(매매)_실거래가_20260531230842.csv
  - 아파트(매매)_실거래가_20260531230857.csv


#### **2. CSV 파일 읽기**

In [3]:
all_shards = []

# 3. CSV 파일 읽기 (인코딩 cp949 우선 시도, 실패 시 utf-8-sig로 재시도)
for file in all_files:
    file_name = os.path.basename(file)
    try:
        try:
            df = pd.read_csv(file, skiprows=15, encoding='cp949', dtype={'번지': str})
        except UnicodeDecodeError:
            df = pd.read_csv(file, skiprows=15, encoding='utf-8-sig', dtype={'번지': str})

        if not df.empty:
            all_shards.append(df)
            print(f"읽기 성공: {file_name} ({len(df)}행)")
    except Exception as e:
        print(f"CSV 파일 읽기 실패 ({file_name}): {e}")

읽기 성공: 아파트(매매)_실거래가_20260531230344.csv (512행)
읽기 성공: 아파트(매매)_실거래가_20260531230404.csv (222행)
읽기 성공: 아파트(매매)_실거래가_20260531230424.csv (499행)
읽기 성공: 아파트(매매)_실거래가_20260531230450.csv (368행)
읽기 성공: 아파트(매매)_실거래가_20260531230511.csv (223행)
읽기 성공: 아파트(매매)_실거래가_20260531230525.csv (299행)
읽기 성공: 아파트(매매)_실거래가_20260531230544.csv (317행)
읽기 성공: 아파트(매매)_실거래가_20260531230556.csv (443행)
읽기 성공: 아파트(매매)_실거래가_20260531230608.csv (497행)
읽기 성공: 아파트(매매)_실거래가_20260531230624.csv (758행)
읽기 성공: 아파트(매매)_실거래가_20260531230637.csv (524행)
읽기 성공: 아파트(매매)_실거래가_20260531230651.csv (395행)
읽기 성공: 아파트(매매)_실거래가_20260531230703.csv (262행)
읽기 성공: 아파트(매매)_실거래가_20260531230716.csv (321행)
읽기 성공: 아파트(매매)_실거래가_20260531230730.csv (565행)
읽기 성공: 아파트(매매)_실거래가_20260531230747.csv (355행)
읽기 성공: 아파트(매매)_실거래가_20260531230806.csv (127행)
읽기 성공: 아파트(매매)_실거래가_20260531230819.csv (257행)
읽기 성공: 아파트(매매)_실거래가_20260531230830.csv (297행)
읽기 성공: 아파트(매매)_실거래가_20260531230842.csv (296행)
읽기 성공: 아파트(매매)_실거래가_20260531230857.csv (201행)


#### **3. 데이터 병합**

In [4]:
# 4. 병합할 파일이 없을 경우 조기 종료
if not all_shards:
    print("병합할 파일이 없습니다.")
else:
    # 전체 파일을 하나의 데이터프레임으로 병합
    raw_df = pd.concat(all_shards, ignore_index=True)
    print(f"합쳐진 초기 데이터 건수: {len(raw_df)}개")

합쳐진 초기 데이터 건수: 7738개


#### **4. 데이터 전처리**

- **4-1. 필요 컬럼 선택 및 도시명 추가**

In [5]:
# 5. 분석에 필요한 컬럼만 선택
keep_cols = ['시군구', '번지', '본번', '부번', '단지명', '전용면적(㎡)',
             '계약년월', '계약일', '거래금액(만원)', '동', '층', '건축년도', '도로명']
raw_df = raw_df[keep_cols]

# 6. 도시명 컬럼 추가 (값: 검단)
raw_df['도시명'] = '검단'

print(f"선택된 컬럼: {list(raw_df.columns)}")
print(f"전체 행 수: {len(raw_df)}개")

선택된 컬럼: ['시군구', '번지', '본번', '부번', '단지명', '전용면적(㎡)', '계약년월', '계약일', '거래금액(만원)', '동', '층', '건축년도', '도로명', '도시명']
전체 행 수: 7738개


- **4-2. 전용면적 타입 변환**

In [6]:
# 7. 전용면적 문자열을 실수(float)로 변환
raw_df['전용면적(㎡)'] = raw_df['전용면적(㎡)'].apply(lambda x: float(x) if str(x).replace('.', '', 1).isdigit() else None)

print(f"전체 행 수: {len(raw_df)}개")
print(raw_df['전용면적(㎡)'].describe().round(2))

전체 행 수: 7738개
count    7738.00
mean       75.76
std        18.09
min        51.30
25%        59.79
50%        84.72
75%        84.99
max       166.29
Name: 전용면적(㎡), dtype: float64


- **4-3. 번지 이상값 탐지 및 제거**

In [7]:
# 8. 번지 이상값 탐지 함수 정의 (날짜 형식으로 잘못 파싱된 값 제거)
# 예: '1983-01-01' 형태(YYYY-MM-DD) 또는 'Jan-83' 형태(영문월-연도)
def is_date_like(val):
    s = str(val)
    parts = s.split('-')
    # YYYY-MM-DD 형태 확인
    if len(parts) == 3 and parts[0].isdigit() and parts[1].isdigit() and parts[2].isdigit():
        if len(parts[0]) == 4 and len(parts[1]) == 2 and len(parts[2]) == 2:
            return True
    # Jan-83 형태 확인
    if len(parts) == 2 and parts[0].isalpha() and parts[1].isdigit():
        if len(parts[0]) == 3 and len(parts[1]) == 2:
            return True
    return False

# 9. 번지 이상값 확인 및 제거
mask = raw_df['번지'].apply(is_date_like)
print(f"번지 이상값 목록: {raw_df[mask]['번지'].unique()}")

before = len(raw_df)
raw_df = raw_df[~mask]
print(f"번지 이상값 제거: {before - len(raw_df)}개 제거 → {len(raw_df)}개 남음")

번지 이상값 목록: <StringArray>
[]
Length: 0, dtype: str
번지 이상값 제거: 0개 제거 → 7738개 남음


#### **5. 결과 저장**

In [8]:
# 10. 전처리 결과 저장
output_path = os.path.join(folder_path, "geomdan_bullo_merged.csv")
raw_df.to_csv(output_path, index=False, encoding='cp949')
print(f"최종 파일 저장 완료: {output_path}")

최종 파일 저장 완료: C:\3_1_DataMining\팀 프로젝트\검단_불로동\geomdan_bullo_merged.csv
